In [1]:
from google.colab import files
uploaded = files.upload()

Saving final_train.csv to final_train.csv
Saving final_validation.csv to final_validation.csv
Saving test_input.csv to test_input.csv


In [2]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [ ]:

train_df = pd.read_csv("final_train.csv")
val_df = pd.read_csv("final_validation.csv")
test_df = pd.read_csv("test_input.csv")


label_encoder = LabelEncoder()
train_df["encoded_label"] = label_encoder.fit_transform(train_df["label"])
val_df["encoded_label"] = label_encoder.transform(val_df["label"])
num_labels = len(label_encoder.classes_)

train_dataset = Dataset.from_pandas(
    train_df[["sentence", "encoded_label"]].rename(columns={"encoded_label": "label"})
)
val_dataset = Dataset.from_pandas(
    val_df[["sentence", "encoded_label"]].rename(columns={"encoded_label": "label"})
)
test_dataset = Dataset.from_pandas(test_df[["sentence"]])

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=160)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

keep_cols_train = ["input_ids", "attention_mask", "label"]
keep_cols_test = ["input_ids", "attention_mask"]
if "token_type_ids" in tokenized_train.column_names:
    keep_cols_train.append("token_type_ids")
    keep_cols_test.append("token_type_ids")

tokenized_train = tokenized_train.remove_columns([c for c in tokenized_train.column_names if c not in keep_cols_train])
tokenized_val = tokenized_val.remove_columns([c for c in tokenized_val.column_names if c not in keep_cols_train])
tokenized_test = tokenized_test.remove_columns([c for c in tokenized_test.column_names if c not in keep_cols_test])


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
    }


use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1.5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=6,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=25,
    report_to="none",
    fp16=use_fp16,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


trainer.train()


val_metrics = trainer.evaluate(tokenized_val)
print("Validation metrics:")
for k, v in val_metrics.items():
    if isinstance(v, (float, np.floating)):
        print(f"   {k}: {v:.4f}")
    else:
        print(f"   {k}: {v}")


pred_output = trainer.predict(tokenized_test)
pred_classes = np.argmax(pred_output.predictions, axis=1)
pred_labels = label_encoder.inverse_transform(pred_classes)

submission_df = test_df.copy()
submission_df["label"] = pred_labels
submission_df = submission_df[["sentence", "label"]]
submission_df.to_csv("predictions.csv", index=False)


1. Loading data...
   Train rows:      12104
   Validation rows: 4035
   Test rows:       4035
2. Loading tokenizer and model (roberta-base)...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


3. Tokenizing datasets...


Map:   0%|          | 0/12104 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

6. Evaluating best checkpoint on validation split...


Validation metrics:
   eval_loss: 0.1460
   eval_accuracy: 0.9698
   eval_f1_macro: 0.8164
   eval_f1_weighted: 0.9692
   eval_runtime: 2.2255
   eval_samples_per_second: 1813.0520
   eval_steps_per_second: 57.0650
   epoch: 6.0000
7. Generating test predictions...
Saved predictions to: predictions.csv
